In [ ]:
with campaign_dates as (

    select
        'ZTAN57' as mrc_desc,
        date '2026-01-01' as start_dt,
        date '2026-01-31' as finish_dt

    union all

    select
        'ZTANTTT63',
        date '2026-02-01',
        date '2026-02-28'

),

first_view as (

    select
        ma.magnit_id,
        ma.mrc_desc,
        min(ma.calc_date::date) as first_view_dt

    from mobapp_act ma

    where ma.metric_id = 1
      and ma.mrc_desc in ('ZTAN57', 'ZTANTTT63')

    group by
        ma.magnit_id,
        ma.mrc_desc

),

subscr as (

    select
        contact_id,
        min(subscr_date_act::date) as first_subscr_dt

    from subscr_status

    group by contact_id

)

select
    cc.client_id,
    fv.magnit_id,
    fv.mrc_desc,
    fv.first_view_dt,
    cd.start_dt,
    cd.finish_dt,
    s.first_subscr_dt,

    case
        when s.first_subscr_dt between cd.start_dt and cd.finish_dt
         and s.first_subscr_dt >= fv.first_view_dt
        then 1
        else 0
    end as subscribed_after_view_flg

from first_view fv

join campaign_dates cd
    on fv.mrc_desc = cd.mrc_desc

join client_cohorts cc
    on fv.magnit_id = cc.magnit_id

left join subscr s
    on cc.client_id = s.contact_id;